In [1]:
import numpy as np
from Env import GridWorld


DISCOUNT_FACTOR = 1 # 请根据不同的任务使用具体discount_factor


class Agent:
    def __init__(self, env):
        self.env = env
        self.V = np.zeros(env.nS)

    def next_best_action(self, s, V):
        action_values = np.zeros(self.env.nA)
        for a in range(self.env.nA):
            for prob, next_state, reward, done in self.env.P[s][a]:
                action_values[a] += prob * (reward + DISCOUNT_FACTOR * V[next_state])
        return np.argmax(action_values), np.max(action_values)

    def optimize(self): # core method
        THETA = 0.0001
        delta = float("inf")
        round_num = 0

        while delta > THETA:
            # 循环条件：只要最大变化量大于阈值，就继续迭代,
            # 当所有状态的价值变化都很小时（≤0.0001），停止迭代
            delta = 0
            # 重置本轮的最大变化量
            # 准备记录本轮中各状态的变化
            print("\nValue Iteration: Round " + str(round_num))
            print(np.reshape(self.V, self.env.shape))
            for s in range(self.env.nS):
                best_action, best_action_value = self.next_best_action(s, self.V)
                delta = max(delta, np.abs(best_action_value - self.V[s]))
                self.V[s] = best_action_value
            round_num += 1

        policy = np.zeros(self.env.nS)
        for s in range(self.env.nS):
            best_action, best_action_value = self.next_best_action(s, self.V)
            policy[s] = best_action

        return policy


# env = GridWorld()
# agent = Agent(env)
# policy = agent.optimize()
# print("\nBest Policy")
# print(np.reshape([env.get_action_name(entry) for entry in policy], env.shape))

# env = GridWorld(wind_prob=.2)
# agent = Agent(env)
# policy = agent.optimize()
# print("\nBest Policy")
# print(np.reshape([env.get_action_name(entry) for entry in policy], env.shape))

In [2]:
from mdp_lib import get_mdp  # 导入同学的MDP

class MDPAdapter:
    """将同学的MDP适配成你的Env格式"""
    
    def __init__(self, mdp_name, **kwargs):
        self.mdp = get_mdp(mdp_name, **kwargs)
        self.nS = self.mdp.n_states
        self.nA = self.mdp.n_actions
        
        # 设置shape用于可视化
        if mdp_name == "gridworld":
            self.shape = (kwargs.get('k', 5), kwargs.get('k', 5))
        elif mdp_name == "chain":
            self.shape = (1, kwargs.get('n', 20))
        elif mdp_name == "gambler":
            goal = kwargs.get('goal', 100)
            self.shape = (1, goal + 1)
        else:
            self.shape = (1, self.nS)
        
        # 构建兼容的P表
        self.P = self._build_transitions()
    
    def _build_transitions(self):
        P = []
        for s in range(self.nS):
            actions = []
            for a in range(self.nA):
                transitions = []
                for next_s in range(self.nS):
                    prob = self.mdp.P[s, a, next_s]
                    if prob > 0:
                        reward = -self.mdp.C[s, a]  # 代价转奖励
                        done = (next_s == self.nS - 1)  # 简化判断
                        transitions.append((prob, next_s, reward, done))
                if len(transitions) == 0:
                    transitions.append((1.0, s, 0, False))
                actions.append(transitions)
            P.append(actions)
        return P
    
    def get_action_name(self, a):
        return f"Action_{a}"

In [6]:
# 测试1: Chain MDP
print("="*60)
print("Testing Chain MDP")
print("="*60)

env = MDPAdapter("chain", n=50, p=0.9, gamma=0.9)
DISCOUNT_FACTOR = 0.9  # 重要：覆盖原来的1
agent = Agent(env)
policy = agent.optimize()

print("\nFinal Value Function:")
print(agent.V.reshape(env.shape))
print("\nFinal Policy (0=RIGHT,1=LEFT):")
print(policy.reshape(env.shape))

Testing Chain MDP

Value Iteration: Round 0
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]]

Value Iteration: Round 1
[[ 0.  -0.1 -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.
  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.
  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.  -1.
  -1.  -1.  -1.  -1.  -1.  -1.  -0.1  0. ]]

Value Iteration: Round 2
[[ 0.      -0.109   -1.17829 -1.9     -1.9     -1.9     -1.9     -1.9
  -1.9     -1.9     -1.9     -1.9     -1.9     -1.9     -1.9     -1.9
  -1.9     -1.9     -1.9     -1.9     -1.9     -1.9     -1.9     -1.9
  -1.9     -1.9     -1.9     -1.9     -1.9     -1.9     -1.9     -1.9
  -1.9     -1.9     -1.9     -1.9     -1.9     -1.9     -1.9     -1.9
  -1.9     -1.9     -1.9     -1.9     -1.9     -1.9     -1.9     -1.171
  -0.109    0.     ]]

Value Iteration: Round 3
[[ 0.         -0.10

In [4]:
print("="*60)
print("Testing Gambler's MDP")
print("="*60)

env = MDPAdapter("gambler", goal=50, p_win=0.4, gamma=0.95)
DISCOUNT_FACTOR = 0.95
agent = Agent(env)
policy = agent.optimize()

print("\nValue Function (first 20):", agent.V[:20])
print("Policy (first 20):", policy[:20])

Testing Gambler's MDP

Value Iteration: Round 0
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0.]]

Value Iteration: Round 1
[[0.        0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.        0.        0.        0.
  0.        0.        0.        0.        0.4       0.4       0.4
  0.4       0.4       0.4       0.4       0.4       0.4       0.4
  0.4       0.4       0.4       0.628     0.628     0.628     0.628
  0.628     0.628     0.75796   0.75796   0.75796   0.8320372 0.8320372
  0.8742612 0.       ]]

Value Iteration: Round 2
[[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.152      0.152      0.152      0.152      0.152
  0.152      0.23864    0.23864    0.23864    0.2880248  0.288024

In [5]:
print("="*60)
print("Testing GridWorld MDP")
print("="*60)

env = MDPAdapter("gridworld", k=4, gamma=0.9, slip_prob=0.1)
DISCOUNT_FACTOR = 0.9
agent = Agent(env)
policy = agent.optimize()

print("\nValue Function:")
print(agent.V.reshape(env.shape))
print("\nPolicy (0=UP,1=DOWN,2=LEFT,3=RIGHT):")
print(policy.reshape(env.shape))

Testing GridWorld MDP

Value Iteration: Round 0
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

Value Iteration: Round 1
[[-1.         -1.03       -1.0309     -1.030927  ]
 [-1.03       -1.0618     -1.062781   -1.06281124]
 [-1.0309     -1.062781   -1.06376686 -0.16379734]
 [-1.030927   -1.06281124 -0.16379734  0.        ]]

Value Iteration: Round 2
[[-1.9018     -1.954135   -1.95646429 -1.95655695]
 [-1.954135   -2.00998414 -2.01178356 -1.2836104 ]
 [-1.95646429 -2.01178356 -1.25829678 -0.18117114]
 [-1.95655695 -1.2836104  -0.18117114  0.        ]]

Value Iteration: Round 3
[[-2.7147601  -2.78328561 -2.78728486 -2.24073639]
 [-2.78328561 -2.85689533 -2.22705411 -1.31929065]
 [-2.78728486 -2.22705411 -1.285807   -0.18358806]
 [-2.24073639 -1.31929065 -0.18358806  0.        ]]

Value Iteration: Round 4
[[-3.44739562 -3.52720862 -3.06057073 -2.29488673]
 [-3.52720862 -3.08235797 -2.26537025 -1.32509276]
 [-3.06057073 -2.26537025 -1.29013619 -0.18396451]
 [-2.29488673 -1.32